# Schur Residual SOHO — train-only CIFAR-100 gates

This notebook tests the post-confusion hypothesis. It reuses the frozen ViT cache and compatible CRT anchor/statistics cache, reports matched raw Ridge, and selects Schur residual rank using only a deterministic training-validation split. It contains no held-out evaluation cell.

In [ ]:
# === Edit this cell only ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/crt-soho'  # push the new Schur commit before running
CHECKPOINT_SOURCE = 'huggingface'
DRIVE_CHECKPOINT_PATH = '/content/drive/MyDrive/T-SOHO/model.safetensors'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_DIR = '/content/tsoho_cifar100_cache'
CRT_GATE_CACHE_DIR = '/content/crt_soho_gate_cache'
OUTPUT_DIR = '/content/schur_residual_gate_outputs'
SEED = 1993
NUM_TASKS = 10
VALIDATION_FRACTION = 0.10
BATCH_SIZE = 128
ANCHOR_BATCH_SIZE = 1024
ANCHOR_DIM = 1024
SYNAPTIC_DEGREE = 300
CODING_LEVEL = 0.30
STATISTICS_DTYPE = 'float32'
# Locked from Phase E; do not reopen their grid in this phase.
ANCHOR_RIDGES = '0.01'
RESIDUAL_RIDGES = '0.1'
COMPLEMENT_RIDGES = '0.1'
RAW_RIDGES = '0.01,0.1,1.0'
RANKS = '16,32,64'
TEMPERATURES = '0.5,1.0'  # applies only to confusion controls
MINIMUM_FULL_GAIN = 0.10
MAXIMUM_LOW_RANK_GAP = 0.50
MINIMUM_PROPOSAL_GAIN = 0.10
MAXIMUM_RELATIVE_SOLVER_RESIDUAL = 1e-4
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'


In [ ]:
import os, shutil, subprocess, sys, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
%cd /content
shutil.rmtree(WORK_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_GIT_URL, WORK_DIR], check=True)
%cd {WORK_DIR}
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
# Resolve checkpoint and CIFAR-100 only for cache creation/preflight.
if CHECKPOINT_SOURCE == 'google_drive':
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
else:
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
import kagglehub
downloaded = Path(kagglehub.dataset_download('zaphat206/cifar-100'))
candidates = [downloaded, *downloaded.rglob('cifar-100')]
cifar_dir = next(p for p in candidates if (p/'train').is_file() and (p/'test').is_file() and (p/'meta').is_file())
CIFAR_ROOT = str(cifar_dir)
print('checkpoint:', CHECKPOINT_PATH)
print('CIFAR-100:', CIFAR_ROOT)


In [ ]:
# Correctness gate.
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_crt_soho_math.py', 'tests/test_crt_gate_runner.py', 'tests/test_experiment_runner.py'], check=True)
if not Path(FEATURE_CACHE_DIR, 'metadata.json').is_file():
    command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--root', CIFAR_ROOT, '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', FEATURE_CACHE_DIR, '--output-dir', f'{OUTPUT_DIR}/feature_extract', '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', str(SEED), '--num-classes', '100', '--num-tasks', str(NUM_TASKS), '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)
else:
    print('Using existing frozen-feature cache:', FEATURE_CACHE_DIR)


In [ ]:
# Train-only Schur gate run. Compatible anchor/statistics cache is reused.
command = [sys.executable, '-u', 'tools/crt_gate_runner.py', '--prepare-cache', '--run-gates', '--proposal-method', 'schur_residual', '--feature-cache-dir', FEATURE_CACHE_DIR, '--gate-cache-dir', CRT_GATE_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--dataset', 'CIFAR-100', '--model-name', 'vit_base_patch16_224', '--num-classes', '100', '--num-tasks', str(NUM_TASKS), '--validation-fraction', str(VALIDATION_FRACTION), '--seed', str(SEED), '--device', 'cuda', '--anchor-dim', str(ANCHOR_DIM), '--synaptic-degree', str(SYNAPTIC_DEGREE), '--coding-level', str(CODING_LEVEL), '--statistics-dtype', STATISTICS_DTYPE, '--anchor-batch-size', str(ANCHOR_BATCH_SIZE), '--raw-ridges', RAW_RIDGES, '--anchor-ridges', ANCHOR_RIDGES, '--residual-ridges', RESIDUAL_RIDGES, '--complement-ridges', COMPLEMENT_RIDGES, '--ranks', RANKS, '--temperatures', TEMPERATURES, '--minimum-full-gain', str(MINIMUM_FULL_GAIN), '--maximum-low-rank-gap', str(MAXIMUM_LOW_RANK_GAP), '--minimum-proposal-gain', str(MINIMUM_PROPOSAL_GAIN), '--maximum-relative-solver-residual', str(MAXIMUM_RELATIVE_SOLVER_RESIDUAL)]
print('Running:', ' '.join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
# Inspect train-validation evidence. Do not launch held-out test.
import json, pandas as pd
report = json.load(open(f'{OUTPUT_DIR}/gate_results.json'))
columns = ['method', 'requested_rank', 'final_effective_rank', 'validation_average_incremental_accuracy', 'validation_final_accuracy', 'persistent_state_bytes', 'solver_relative_residual_max', 'retained_correction_energy']
table = pd.DataFrame(report['candidates'])
display(table[[column for column in columns if column in table]].sort_values('validation_average_incremental_accuracy', ascending=False))
print(json.dumps({'selected_raw_ridge': report['selected_raw_ridge'], 'selected_proposal': report.get('selected_proposal'), 'gates': report['gates'], 'final_subspace_diagnostics': report.get('final_subspace_diagnostics'), 'held_out_test_authorized': report['held_out_test_authorized']}, indent=2))
print('STOP HERE and send gate_results.json for review.')


In [ ]:
archive = '/content/schur_residual_gate_results.zip'
subprocess.run(['zip', '-r', archive, OUTPUT_DIR], check=True)
from google.colab import files
files.download(archive)
